In [1]:
import pandas as pd
import numpy as np
import altair as alt
import seaborn as sns
from pybliometrics import scopus
from main import Corpus
import Bibliography_Parse
import geopandas as gpd
import matplotlib.pyplot as plt
import gpdvega
import multiprocessing
from geopy.geocoders import Photon
color_dict={'Astrobiology': '#2FD03D', 'Bioastronautics': '#3D2FD0', 'Space Bioprocess Engineering': '#D03D2F'}
color_dict_2={'AB': '#2FD03D', 'BA': '#3D2FD0', 'SBE': '#D03D2F'}

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('gpd_to_values')

In [2]:
author_ids = pd.read_csv('unique_authors')['0']
author_ids

0         8397364800
1        36664765600
2        35177423100
3         7005325946
4         7409615288
            ...     
49718    57215592822
49719     8424066900
49720     8706064700
49721     8644474800
49722    55584445800
Name: 0, Length: 49723, dtype: int64

In [3]:
bioscience_bibliography = pd.read_csv('bioscience_bibliography')
bioscience_bibliography.author_ids = bioscience_bibliography.author_ids.str.split(';')
bioscience_bibliography.author_afids = bioscience_bibliography.author_afids.str.split(';')
bioscience_bibliography.afid = bioscience_bibliography.afid.str.split(';')
bioscience_bibliography = bioscience_bibliography[['author_ids', 'author_afids', 'afid']]
# bioscience_bibliography.author_afids.apply(lambda x: len(x) if type(x)==list else 1)
bioscience_bibliography

,author_ids,author_afids,afid
0,"[56993931000, 54974731000, 58153269000, 700494...","[60027272, 60012708-124841290, 129393708, 6002...","[60170182, 60072546, 60031581, 60027272, 60014..."
1,"[56487197100, 58146986100, 6507732731, 1360595...","[60028039, 60136210-60073837, 60028039, 60028039]","[60136210, 60073837, 60028039]"
2,"[57222579646, 57211144665, 57192302909, 222342...","[60019616, 60019616, 60030637, 60019616, 60019...","[60030637, 60019616, 60001439, 60000481]"
3,[7102359712],[60007202-113582880],"[60007202, 113582880]"
4,"[57212383059, 57218845028, 57278545200, 571915...","[121595840-60011976, 121595840, 121595840, 121...","[60011976, 128131936, 121595840, 115293926]"
...,...,...,...
24784,[35520941200],NaN,NaN
24785,NaN,NaN,NaN
24786,[7404152751],NaN,NaN
24787,[6508021729],NaN,NaN


In [77]:
# scopus.AffiliationRetrieval(110785288)._profile['@parent']

# pd.DataFrame(scopus.AffiliationSearch('af-id(110785288)').affiliations)

# scopus.AffiliationRetrieval(113587821).affiliation_name
# scopus.AffiliationRetrieval(113587821).self_link

dict_keys(['@affiliation-id', 'status', 'date-created', 'preferred-name', 'sort-name', 'name-variant', 'address'])

In [7]:
def to_1D(series):
    flattened = []
    for element in series:
        if isinstance(element, list):
            flattened.extend(element)
        elif isinstance(element, np.ndarray):
            flattened.extend(element.tolist())
    return pd.Series(flattened)


set(to_1D(bioscience_bibliography.afid))


{'60000088',
 '113075111',
 '60114739',
 '60020990',
 '60015615',
 '60072836',
 '60020321',
 '60112425',
 '60072236',
 '60020096',
 '60004814',
 '60007183',
 '60007103',
 '60003079',
 '60072296',
 '60025352',
 '60022286',
 '60178602',
 '128987739',
 '126274105',
 '100319158',
 '60012162',
 '60012689',
 '110685232',
 '60028925',
 '60017252',
 '60019359',
 '60005384',
 '60101299',
 '60030426',
 '60011733',
 '60156343',
 '100818130',
 '60021323',
 '60085627',
 '60020730',
 '60017681',
 '112884793',
 '60033165',
 '118445492',
 '114402415',
 '60103895',
 '60006404',
 '60021542',
 '60000765',
 '108418597',
 '60003531',
 '60139009',
 '60008724',
 '60174564',
 '60008208',
 '106639316',
 '60025884',
 '60077572',
 '60002445',
 '60072960',
 '60024878',
 '60003970',
 '121573339',
 '118496271',
 '123196386',
 '60012771',
 '125190635',
 '60032499',
 '107103023',
 '107572501',
 '119562782',
 '60030364',
 '60030114',
 '100341857',
 '118747359',
 '121099540',
 '60032895',
 '60070029',
 '60005634',
 '12

In [5]:
bioscience_bibliography.afid.apply(pd.Series).iloc[:,0].value_counts(normalize = True)

60004179     0.019211
60006711     0.018427
60069097     0.017381
60009037     0.010716
60016296     0.009888
               ...   
60031447     0.000044
106536368    0.000044
60045954     0.000044
112895248    0.000044
115789797    0.000044
Name: 0, Length: 4775, dtype: float64

In [45]:
scopus.AffiliationRetrieval(60007202).__dir__()


['_view',
 '_refresh',
 '_cache_file_path',
 '_mdate',
 '_json',
 '_profile',
 '__module__',
 'address',
 'affiliation_name',
 'author_count',
 'city',
 'country',
 'date_created',
 'document_count',
 'eid',
 'identifier',
 'name_variants',
 'org_domain',
 'org_type',
 'org_URL',
 'postal_code',
 'scopus_affiliation_link',
 'self_link',
 'search_link',
 'state',
 'status',
 'sort_name',
 'url',
 '__init__',
 '__str__',
 '__doc__',
 'get_cache_file_age',
 'get_cache_file_mdate',
 'get_key_remaining_quota',
 'get_key_reset_time',
 '__dict__',
 '__weakref__',
 '__repr__',
 '__hash__',
 '__getattribute__',
 '__setattr__',
 '__delattr__',
 '__lt__',
 '__le__',
 '__eq__',
 '__ne__',
 '__gt__',
 '__ge__',
 '__new__',
 '__reduce_ex__',
 '__reduce__',
 '__subclasshook__',
 '__init_subclass__',
 '__format__',
 '__sizeof__',
 '__dir__',
 '__class__']

In [6]:
scopus.AffiliationRetrieval(60004179)._profile

{'@affiliation-id': '60004179',
 'status': 'update',
 'date-created': {'@day': '02', '@month': '02', '@year': '2008'},
 'preferred-name': {'@source': 'internal-ani',
  '$': 'NASA Ames Research Center'},
 'sort-name': 'NASA Ames Research Center',
 'name-variant': [{'@doc-count': '29404',
   '@source': 'internal-ani',
   '$': 'Nasa Ames Research Center'},
  {'@doc-count': '3303',
   '@source': 'internal-ani',
   '$': 'Ames Research Center'},
  {'@doc-count': '1919',
   '@source': 'internal-ani',
   '$': 'Nasa-ames Research Center'},
  {'@doc-count': '1062', '@source': 'internal-ani', '$': 'Nasa'},
  {'@doc-count': '841',
   '@source': 'internal-ani',
   '$': 'Nasa Ames Research Cent'},
  {'@doc-count': '656',
   '@source': 'internal-ani',
   '$': 'Nasa/ames Research Center'},
  {'@doc-count': '642', '@source': 'internal-ani', '$': 'Ames Research Cent'},
  {'@doc-count': '367', '@source': 'internal-ani', '$': 'Nasa Ames'},
  {'@doc-count': '317', '@source': 'internal-ani', '$': 'Nasa Rese

In [25]:
pd.DataFrame(scopus.AuthorRetrieval(7004212771).affiliation_history)

,id,parent,type,relationship,afdispname,preferred_name,parent_preferred_name,country_code,country,address_part,city,state,postal_code,org_domain,org_URL
0,110785688,60027950.0,dept,author,None,Department of Chemical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
1,60027950,NaN,parent,author,None,Carnegie Mellon University,None,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
2,113587821,60027950.0,dept,author,None,Department of Mechanical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
3,60090776,NaN,parent,author,None,"National Energy Technology Laboratory, Pittsburgh",None,usa,United States,626 Cochrans Mill Road,Pittsburgh,PA,15236-0940,netl.doe.gov,https://netl.doe.gov/
4,113613418,60027950.0,dept,author,None,Department of Materials Science and Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
5,109531166,60027950.0,dept,author,None,Department of Chemical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
6,108790916,60027950.0,dept,author,None,Department of Chemical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
7,108336146,60027950.0,dept,author,None,Department of Chemical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/
8,105243609,60023143.0,dept,author,None,Department of Chemistry,Tufts University,usa,United States,419 Boston Avenue,Medford,MA,02155-5801,tufts.edu,https://www.tufts.edu/
9,104415945,60027950.0,dept,author,None,Department of Chemical Engineering,Carnegie Mellon University,usa,United States,5000 Forbes Avenue,Pittsburgh,PA,15213-3890,cmu.edu,https://www.cmu.edu/


In [22]:
bioscience_bibliography = pd.read_csv('bioscience_bibliography')
biolists = bioscience_bibliography.afid.dropna().str.split(';')
numpy_list = biolists.to_numpy()
flat_lst = [item for sublist in numpy_list if sublist is not None for item in sublist]
# unique_afids = list(set(flat_lst))
unique_afids = [int(x) for x in flat_lst]
unique_afids

[60170182,
 60072546,
 60031581,
 60027272,
 60014340,
 60012708,
 60011019,
 60006288,
 60004179,
 60003892,
 60000221,
 129393708,
 129393671,
 128944298,
 127131603,
 124841290,
 119559805,
 60136210,
 60073837,
 60028039,
 60030637,
 60019616,
 60001439,
 60000481,
 60007202,
 113582880,
 60011976,
 128131936,
 121595840,
 115293926,
 60032991,
 60027272,
 60017293,
 60172343,
 60017293,
 60017293,
 60105942,
 60000964,
 60208589,
 60122734,
 60094055,
 60009358,
 115043608,
 60031159,
 60019544,
 60003763,
 60025038,
 124841290,
 60019611,
 60077190,
 60029681,
 60028717,
 60022457,
 60017246,
 60012556,
 60005881,
 60002238,
 60002128,
 60024324,
 60002746,
 60002243,
 60019586,
 60012464,
 60007798,
 60000765,
 60170182,
 60162625,
 60106381,
 60012708,
 60011460,
 60006222,
 60004636,
 60004179,
 128664260,
 101951639,
 60172345,
 60003771,
 60002798,
 125643328,
 60111154,
 60008293,
 60007798,
 60069097,
 60003172,
 60032006,
 60025038,
 60009507,
 60008293,
 60028717,
 60012

In [27]:
from collections import Counter
df = pd.DataFrame(Counter(unique_afids).most_common(1000))
df['affilname'] = df.apply(lambda x: scopus.AffiliationRetrieval(x[0], subscriber='TRUE', view='STANDARD').affiliation_name, axis=1)
df
df.to_excel('affiliation_count.xlsx')

## Affiliation

In [24]:
pd.DataFrame(Counter(unique_afids).most_common(500))

,0,1
0,60004179,1345
1,60006711,1173
2,60009037,836
3,60007798,790
4,60069097,506
...,...,...
495,60001970,21
496,60022558,21
497,60022020,21
498,60098668,21


In [ ]:
ab = scopus.AbstractRetrieval("2-s2.0-85068268027", view='FULL')
ab.subject_areas

subject_areas = {}

for author in authors:
    scopus_retrieval = scopus.AbstractRetrieval("2-s2.0-85068268027", view='FULL')
    

In [7]:
%%time

unique_author_df = pd.DataFrame(index=author_ids, columns=['eids', 'affiliation_history', 'affiliation_current', 'h_index', 'name', 'document_count', 'citation_count'])

for i, author_id in enumerate(author_ids[-100:]):
    # ['h_index', 'affiliation_history', 'affiliation_current', 'eid', 'indexed_name']
    scopus_retrieval = scopus.AuthorRetrieval(author_id, view='ENHANCED')
    unique_author_df.iloc[i, 1] = pd.DataFrame(scopus_retrieval.affiliation_history)
    unique_author_df.iloc[i, 2] = pd.DataFrame(scopus_retrieval.affiliation_current)
    unique_author_df.iloc[i, 0] = scopus_retrieval.eid
    unique_author_df.iloc[i, 3] = scopus_retrieval.h_index
    unique_author_df.iloc[i, 4] = scopus_retrieval.indexed_name
    unique_author_df.iloc[i, 5] = scopus_retrieval.document_count
    unique_author_df.iloc[i, 6] = scopus_retrieval.citation_count
    

unique_author_df



CPU times: user 992 ms, sys: 48.2 ms, total: 1.04 s
Wall time: 37.6 s


,eids,affiliation_history,affiliation_current,h_index,name,document_count,citation_count
0,,,,,,,
8397364800,9-s2.0-35579551400,id parent type relationship...,id parent type relationship afdi...,15,Idkaidek N.,68,717
36664765600,9-s2.0-37020917700,id parent type relationship afdi...,id parent type relationship afdi...,1,Ma S.,1,2
35177423100,9-s2.0-7003664723,id parent type relationshi...,id parent type relationship afdi...,12,Kangas P.,52,698
7005325946,9-s2.0-57251129200,id parent type relationship afdisp...,id parent type relationship afdisp...,1,Schmidt H.,2,2
7409615288,9-s2.0-57209639786,id parent type relationship afdi...,id parent type relationship afdi...,0,Stefanski P.,1,0
...,...,...,...,...,...,...,...
57215592822,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8424066900,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8706064700,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
%%time

def fetch_author_data(i, author_id, shared_dict):
    scopus_retrieval = scopus.AuthorRetrieval(author_id, view='ENHANCED')
    shared_dict[author_id] = {
        'eids': scopus_retrieval.eid,
        'affiliation_history': pd.DataFrame(scopus_retrieval.affiliation_history),
        'affiliation_current': pd.DataFrame(scopus_retrieval.affiliation_current),
        'h_index': scopus_retrieval.h_index,
        'name': scopus_retrieval.indexed_name,
        'document_count': scopus_retrieval.document_count,
        'citation_count': scopus_retrieval.citation_count
    }

def multiprocess_authors(author_ids):
    unique_author_df = pd.DataFrame(index=author_ids, columns=['eids', 'affiliation_history', 'affiliation_current', 'h_index', 'name', 'document_count', 'citation_count'])
    author_ids_chunk = author_ids[1000:2000]

    manager = multiprocessing.Manager()
    shared_dict = manager.dict()

    processes = []
    for i, author_id in enumerate(author_ids_chunk):
        process = multiprocessing.Process(target=fetch_author_data, args=(i, author_id, shared_dict))
        processes.append(process)
        process.start()

    for process in processes:
        process.join()

    for author_id, result in shared_dict.items():
        unique_author_df.loc[author_id] = [
            result['eids'],
            result['affiliation_history'],
            result['affiliation_current'],
            result['h_index'],
            result['name'],
            result['document_count'],
            result['citation_count']
        ]

    return unique_author_df

unique_author_df = multiprocess_authors(author_ids)
unique_author_df[-100:]
